# Assignment 2 · Parte 1 – Scraping de decretos de emergencia

**Grupo 10** · Temporada asignada: **1 de enero al 31 de mayo de 2023**

Pregunta del trabajo: **¿El Estado declara emergencia donde más llueve?**

En este notebook se reúnen los decretos supremos de la PCM que declaran (o prorrogan) el Estado de Emergencia por lluvias, usando el buscador de normas de gob.pe. Como esa página arma los resultados con JavaScript, usamos **Selenium**. Para la página individual de cada norma se utilizará con `requests` + `BeautifulSoup`.

## 0. Comprobar el entorno

Igual que en clase: esta celda **revisa** qué falta. Si falta algo, descomenta la línea del `%pip`, ejecútala y reinicia el kernel.

In [1]:
import sys

faltan = []
for modulo, paquete in [("selenium", "selenium"), ("bs4", "beautifulsoup4"),
                        ("lxml", "lxml"), ("pandas", "pandas"),
                        ("requests", "requests")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)

print("Entorno:", sys.executable)
if faltan:
    print("\nFaltan:", " ".join(faltan))
    print(f"   %pip install {' '.join(faltan)}")
else:
    import selenium
    print("Todo instalado. selenium", selenium.__version__)

# %pip install selenium beautifulsoup4 lxml requests pandas

Entorno: c:\Users\marco\anaconda3\python.exe
Todo instalado. selenium 4.49.0


In [2]:
import os
import re
import time
import calendar
from urllib import robotparser

import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException

pd.set_option("display.max_colwidth", 120)

# Carpeta donde se guardan los CSV (dentro de assignment_2/)
os.makedirs("datos", exist_ok=True)

# Parámetros de nuestra temporada (Grupo 10 → 2023)
ANIO = 2023
MESES = [1, 2, 3, 4, 5]          # enero a mayo
NOMBRE_MES = {1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo"}

HEADERS = {"User-Agent": "Mozilla/5.0"}

## 1. Leer el `robots.txt` de gob.pe

Antes de automatizar nada revisamos qué pide el sitio. Imprimimos el archivo completo y, además, lo analizamos con `urllib.robotparser` para **comprobar con código** (no a ojo) si `/busquedas` y las páginas de normas están permitidas.

In [3]:
robots = requests.get("https://www.gob.pe/robots.txt", headers=HEADERS, timeout=30)
print("Código:", robots.status_code)
print("-" * 60)
print(robots.text)

Código: 200
------------------------------------------------------------
# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



In [4]:
# Comprobación automática con el parser oficial de Python
rp = robotparser.RobotFileParser()
rp.parse(robots.text.splitlines())

url_busqueda_prueba = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
                       "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")
url_norma_prueba = "https://www.gob.pe/institucion/pcm/normas-legales/1-algo"

print("¿/busquedas permitido para '*'?     ", rp.can_fetch("*", url_busqueda_prueba))
print("¿Página de norma permitida para '*'?", rp.can_fetch("*", url_norma_prueba))

# Reglas y Crawl-delay por bot (para ver qué les piden a los demás)
print("\nLíneas Disallow:")
for linea in robots.text.splitlines():
    if linea.strip().lower().startswith("disallow"):
        print("  ", linea.strip())

print("\nCrawl-delay declarados:")
bot_actual = None
for linea in robots.text.splitlines():
    l = linea.strip()
    if l.lower().startswith("user-agent"):
        bot_actual = l.split(":", 1)[1].strip()
    elif l.lower().startswith("crawl-delay"):
        print(f"   {bot_actual}: {l.split(':', 1)[1].strip()} segundos")

¿/busquedas permitido para '*'?      True
¿Página de norma permitida para '*'? True

Líneas Disallow:
   Disallow: /admin/
   Disallow: /*?sheet=
   Disallow: /*&sheet=
   Disallow: /
   Disallow: /admin/
   Disallow: /*?sheet=
   Disallow: /*&sheet=
   Disallow: /admin/
   Disallow: /*?sheet=
   Disallow: /*&sheet=

Crawl-delay declarados:
   GPTBot: 5 segundos
   OAI-SearchBot: 2 segundos


### ¿Qué dice el `robots.txt`?

- **Qué prohíbe:** las líneas `Disallow` listadas arriba son las rutas que gob.pe pide no recorrer automáticamente: /admin/, /*?sheet=, /*&sheet=.
- **¿`/busquedas` está permitida?** Según `robotparser`, `can_fetch("*", ...)` devolvió **True**. Las páginas de normas (`/institucion/pcm/normas-legales/...`) también devolvieron **True**.
- **¿Por qué igual hacemos pausas?** Aunque nuestra ruta esté permitida, el archivo pide un `Crawl-delay` de *5* segundos a otros bots (*GPTBot*). Eso muestra que el sitio quiere que **nadie** le haga pedidos seguidos: es un servidor público del Estado y cada búsqueda la ejecuta su servidor. Por eso dejamos **≥ 2 s entre páginas de búsqueda** y **1 s entre páginas de normas**; así nos comportamos como un usuario y no como un ataque (y evitamos que nos bloqueen).

## 2. Abrir el navegador (misma función que en clase)

`HEADLESS = False` para ver la ventana mientras exploramos; para la corrida completa se puede poner en `True`.

In [5]:
HEADLESS = False

def abrir_navegador():
    opciones = Options()
    if HEADLESS:
        opciones.add_argument("--headless=new")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opciones)

### URL de búsqueda por mes

La URL base más `&desde=DD-MM-AAAA&hasta=DD-MM-AAAA`. Usamos `calendar.monthrange` para obtener el último día de cada mes (así febrero queda en 28 y no hay que escribirlo a mano).

In [6]:
URL_BASE = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
            "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")

def url_mes(anio, mes, pagina=1):
    ultimo_dia = calendar.monthrange(anio, mes)[1]
    url = f"{URL_BASE}&desde=01-{mes:02d}-{anio}&hasta={ultimo_dia:02d}-{mes:02d}-{anio}"
    if pagina > 1:
        url += f"&page={pagina}"
    return url

for m in MESES:
    print(NOMBRE_MES[m], "→", url_mes(ANIO, m))

enero → https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2023&hasta=31-01-2023
febrero → https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-02-2023&hasta=28-02-2023
marzo → https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-03-2023&hasta=31-03-2023
abril → https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-04-2023&hasta=30-04-2023
mayo → https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-05-2023&hasta=31-05-2023


### Esperar bien: `WebDriverWait`

No usamos solo `time.sleep()`. Esperamos **hasta que** aparezca alguno de estos dos indicios:
- al menos un `<article>` (hay resultados), o
- el texto "N Resultados" (cubre el caso de un mes con 0 resultados, donde nunca aparecería un `article`).

Después dejamos 2 segundos de margen porque la página sigue acomodándose.

In [7]:
PATRON_TOTAL = re.compile(r"([\d.,]+)\s+resultados?", re.IGNORECASE)

def pagina_lista(driver):
    # Condición para WebDriverWait: hay artículos o ya se ve el contador.
    if driver.find_elements(By.TAG_NAME, "article"):
        return True
    cuerpo = driver.find_element(By.TAG_NAME, "body").text
    return bool(PATRON_TOTAL.search(cuerpo)) or "no se encontraron" in cuerpo.lower()


def cargar(driver, url, espera_max=30):
    driver.get(url)
    WebDriverWait(driver, espera_max).until(pagina_lista)
    time.sleep(2)                      # margen extra
    if "acceso restringido" in driver.title.lower():
        raise RuntimeError("gob.pe devolvió 'Acceso restringido'. Esperar unos minutos y reintentar.")


def leer_total(driver):
    # Lee el número de 'N Resultados' que muestra la página.
    cuerpo = driver.find_element(By.TAG_NAME, "body").text
    m = PATRON_TOTAL.search(cuerpo)
    if not m:
        return 0
    return int(m.group(1).replace(".", "").replace(",", ""))

### Probar con un solo mes (febrero 2023)

Primero un mes, como pide el paso 2, y miramos un `article` por dentro antes de escribir la función que extrae.

In [8]:
navegador = abrir_navegador()
cargar(navegador, url_mes(ANIO, 2))

print("Título de la pestaña:", navegador.title)
print("Resultados totales  :", leer_total(navegador))

articulos = navegador.find_elements(By.TAG_NAME, "article")
print("Artículos en pantalla:", len(articulos))

Título de la pestaña: Plataforma del Estado Peruano
Resultados totales  : 9
Artículos en pantalla: 9


In [9]:
# ¿Qué trae un article? Miramos su texto y sus enlaces
primero = articulos[0]
print(primero.text)
print("-" * 60)
for a in primero.find_elements(By.TAG_NAME, "a"):
    print(repr(a.text[:80]), "→", a.get_attribute("href"))

PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 027-2023-PCM
Publicado: 24 de febrero de 2023
 Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Padre Abad del departamento de Ucayali 
Descargar archivo
------------------------------------------------------------
'Decreto Supremo N.° 027-2023-PCM' → https://www.gob.pe/institucion/pcm/normas-legales/5569225-027-2023-pcm
'Descargar archivo' → https://cdn.www.gob.pe/uploads/document/file/6341104/5569225-ds-n-027-2023-pcm.pdf


### Función que convierte un `article` en un diccionario

Igual que `leer_oferta()` en Bumeran. Evitamos clases CSS generadas y usamos cosas estables:

- **número**: el texto del enlace principal (el primer `<a>` con texto que apunta a una norma);
- **fecha**: la línea que empieza con `Publicado:`;
- **título**: el párrafo debajo de la fecha (primer `<p>` que no es la fecha; si no hay `<p>`, la línea siguiente a la fecha);
- **enlace**: el `href` del enlace principal.

In [10]:
def leer_resultado(article):
    lineas = [l.strip() for l in article.text.split("\n") if l.strip()]

    # Enlace principal: primer <a> con texto (preferimos los que apuntan a normas)
    enlaces = [a for a in article.find_elements(By.TAG_NAME, "a") if a.text.strip()]
    principal = next((a for a in enlaces if "norma" in (a.get_attribute("href") or "")),
                     enlaces[0] if enlaces else None)
    numero = principal.text.strip() if principal else None
    enlace = principal.get_attribute("href") if principal else None

    # Fecha: la línea que empieza con "Publicado:"
    linea_fecha = next((l for l in lineas if l.startswith("Publicado")), None)
    fecha = linea_fecha.replace("Publicado:", "").strip() if linea_fecha else None

    # Título: párrafo debajo de la fecha
    parrafos = [p.text.strip() for p in article.find_elements(By.TAG_NAME, "p") if p.text.strip()]
    titulo = next((p for p in parrafos if not p.startswith("Publicado") and p != numero), None)
    if titulo is None and linea_fecha in lineas:
        i = lineas.index(linea_fecha)
        titulo = lineas[i + 1] if i + 1 < len(lineas) else None

    return {"numero": numero, "fecha": fecha, "titulo": titulo, "enlace": enlace}


pd.DataFrame([leer_resultado(a) for a in articulos[:5]])

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 027-2023-PCM,24 de febrero de 2023,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Padre Abad del departamento de Uca...,https://www.gob.pe/institucion/pcm/normas-legales/5569225-027-2023-pcm
1,Decreto Supremo N.° 025-2023-PCM,18 de febrero de 2023,Decreto Supremo que prorroga el Estado de Emergencia en la provincia de Condorcanqui y en los distritos de Imaza y A...,https://www.gob.pe/institucion/pcm/normas-legales/5569222-025-2023-pcm
2,Decreto Supremo N.° 024-2023-PCM,17 de febrero de 2023,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Allauca, Cacra, Catahuasi, Huangascar, Linch...",https://www.gob.pe/institucion/pcm/normas-legales/5569220-024-2023-pcm
3,Decreto Supremo N.° 023-2023-PCM,16 de febrero de 2023,Decreto Supremo que declara el Estado de Emergencia en las provincias de Putumayo y Mariscal Ramón Castilla del depa...,https://www.gob.pe/institucion/pcm/normas-legales/5569218-023-2023-pcm
4,Decreto Supremo N.° 022-2023-PCM,13 de febrero de 2023,"Decreto Supremo que prorroga el Estado de Emergencia declarado en el departamento de Lima, en la Provincia Constituc...",https://www.gob.pe/institucion/pcm/normas-legales/5569214-022-2023-pcm


In [11]:
navegador.quit()     # cerramos el navegador de exploración

## 3. Todos los meses de la temporada

La función recorre un mes completo. Si un mes tiene más resultados de los que caben en una página, sigue con la página siguiente (`&page=2`, `&page=3`, ...) hasta haber extraído el total o hasta que una página no traiga nada nuevo.

- **Pausa ≥ 2 s** entre cada página (`time.sleep(ESPERA)`).
- `try / finally` para cerrar Chrome siempre.
- Las fechas se extraen en el momento: si guardáramos los `WebElement` y la página cambia, saldría `StaleElementReferenceException`.

In [12]:
ESPERA = 2.5     # segundos entre páginas (mínimo pedido: 2)

def scrapear_mes(driver, anio, mes, max_paginas=20):
    cargar(driver, url_mes(anio, mes, 1))
    total = leer_total(driver)
    filas, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        if pagina > 1:
            time.sleep(ESPERA)
            cargar(driver, url_mes(anio, mes, pagina))

        nuevos = 0
        for art in driver.find_elements(By.TAG_NAME, "article"):
            fila = leer_resultado(art)
            if fila["enlace"] in vistos:
                continue
            vistos.add(fila["enlace"])
            fila["mes"] = NOMBRE_MES[mes]
            filas.append(fila)
            nuevos += 1

        print(f"   {NOMBRE_MES[mes]} · página {pagina}: {nuevos} nuevos (acumulado {len(filas)}/{total})")
        if len(filas) >= total or nuevos == 0:
            break

    return total, filas


resumen, todas = [], []
navegador = abrir_navegador()
try:
    for mes in MESES:
        total, filas = scrapear_mes(navegador, ANIO, mes)
        resumen.append({"mes": NOMBRE_MES[mes],
                        "resultados_totales": total,
                        "resultados_extraidos": len(filas)})
        todas.extend(filas)
        time.sleep(ESPERA)            # pausa también entre meses
finally:
    navegador.quit()

decretos = pd.DataFrame(todas)
decretos.shape

   enero · página 1: 11 nuevos (acumulado 11/11)
   febrero · página 1: 9 nuevos (acumulado 9/9)
   marzo · página 1: 14 nuevos (acumulado 14/14)
   abril · página 1: 11 nuevos (acumulado 11/11)
   mayo · página 1: 8 nuevos (acumulado 8/8)


(53, 5)

## 4. Verificación: ¿extrajimos todo?

`resultados_totales` (lo que dice la página) debe ser igual a `resultados_extraidos` (lo que leímos) en **todos** los meses.

In [13]:
verificacion = pd.DataFrame(resumen)
verificacion["coincide"] = verificacion["resultados_totales"] == verificacion["resultados_extraidos"]
print(verificacion.to_string(index=False))
print()
print("Total extraído:", verificacion["resultados_extraidos"].sum())

assert verificacion["coincide"].all(), "Algún mes no coincide: revisar paginación/selectores antes de seguir."
print("✔ Coinciden en todos los meses")

    mes  resultados_totales  resultados_extraidos  coincide
  enero                  11                    11      True
febrero                   9                     9      True
  marzo                  14                    14      True
  abril                  11                    11      True
   mayo                   8                     8      True

Total extraído: 53
✔ Coinciden en todos los meses


## 5. Limpieza: duplicados y normas de otras instituciones

In [14]:
antes = len(decretos)
decretos = decretos.drop_duplicates(subset="enlace").reset_index(drop=True)
print(f"Duplicados eliminados: {antes - len(decretos)}  →  quedan {len(decretos)}")

Duplicados eliminados: 0  →  quedan 53


In [15]:
es_pcm = decretos["enlace"].str.contains("/institucion/pcm/", na=False)

print("Normas que NO son de la PCM (se eliminan):")
print(decretos.loc[~es_pcm, ["numero", "enlace"]].to_string(index=False))

antes = len(decretos)
decretos = decretos[es_pcm].reset_index(drop=True)
print(f"\nSe quitaron {antes - len(decretos)} normas de otras instituciones. Quedan {len(decretos)}.")

Normas que NO son de la PCM (se eliminan):
                         numero                                                                      enlace
Resolución N.° 0209-2023-TCE-S1 https://www.gob.pe/institucion/oece/normas-legales/3850417-0209-2023-tce-s1
Resolución N.° 0090-2023-TCE-S1 https://www.gob.pe/institucion/oece/normas-legales/3837479-0090-2023-tce-s1
Resolución N.° 1665-2023-TCE-S2 https://www.gob.pe/institucion/oece/normas-legales/4105780-1665-2023-tce-s2
Resolución N.° 1820-2023-TCE-S4 https://www.gob.pe/institucion/oece/normas-legales/4157393-1820-2023-tce-s4

Se quitaron 4 normas de otras instituciones. Quedan 49.


## 6. Título completo desde la página de cada norma

En la lista los títulos vienen cortados con "...". La página de cada norma **sí trae el HTML completo**, así que no hace falta Selenium: `requests` + `BeautifulSoup`, con `User-Agent` de navegador y **1 segundo de pausa** entre páginas.

El título está en `<div class="description">`. Al final trae texto extra del botón de descarga (*"DS N° ... PDF ... Descargar"*). Lo limpiamos cortando desde donde empieza ese bloque, pero solo si lo que queda sigue siendo un título largo (para no cortar de más).

In [16]:
PATRON_EXTRA = re.compile(r"\s+(D\.?\s?S\.?|Decreto Supremo)\s+N[°º.]?.*?(PDF|Descargar).*$",
                          re.IGNORECASE | re.DOTALL)

def limpiar_titulo(texto):
    texto = re.sub(r"\s+", " ", texto).strip()
    limpio = PATRON_EXTRA.sub("", texto).strip()
    return limpio if len(limpio) >= 40 else texto


def titulo_completo(enlace):
    r = requests.get(enlace, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        return None
    sopa = BeautifulSoup(r.text, "lxml")
    div = sopa.find("div", class_="description")
    if div is None:
        return None
    return limpiar_titulo(div.get_text(" ", strip=True))


# Probar con una sola norma antes de recorrer todas
print(decretos.loc[0, "titulo"])
print("→", titulo_completo(decretos.loc[0, "enlace"]))

Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de las provincias del departamento de Puno, por peligro inminente ante déficit hídrico
→ Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de las provincias del departamento de Puno, por peligro inminente ante déficit hídrico


In [17]:
completos = []
for i, enlace in enumerate(decretos["enlace"], start=1):
    completos.append(titulo_completo(enlace))
    if i % 10 == 0:
        print(f"   {i}/{len(decretos)} normas leídas")
    time.sleep(1)                       # pausa pedida entre páginas

decretos["titulo_completo"] = completos

sin_titulo = decretos["titulo_completo"].isna().sum()
print("Normas sin título completo:", sin_titulo)
# Si alguna falló, usamos el título corto para no perder la fila (y lo reportamos)
decretos["titulo_completo"] = decretos["titulo_completo"].fillna(decretos["titulo"])

   10/49 normas leídas
   20/49 normas leídas
   30/49 normas leídas
   40/49 normas leídas
Normas sin título completo: 0


In [18]:
# Verificación: el título completo debe ser más largo que el cortado y no terminar en "..."
decretos["largo_corto"] = decretos["titulo"].str.len()
decretos["largo_completo"] = decretos["titulo_completo"].str.len()
print("Títulos completos que siguen terminando en '...':",
      decretos["titulo_completo"].str.endswith("...").sum())
decretos[["numero", "largo_corto", "largo_completo"]].head(10)

Títulos completos que siguen terminando en '...': 0


,numero,largo_corto,largo_completo
0,Decreto Supremo N.° 017-2023-PCM,159,159
1,Decreto Supremo N.° 016-2023-PCM,109,109
2,Decreto Supremo N.° 014-2023-PCM,203,226
3,Decreto Supremo N.° 010-2023-PCM,104,104
4,Directiva N.° 005-2015-PCM/SGRD,180,275
5,Directiva N.° 004-2015-PCM/SGRD,174,269
6,Decreto Supremo N.° 009-2023-PCM,203,409
7,Decreto Supremo N.° 008-2023-PCM,203,331
8,Decreto Supremo N.° 007-2023-PCM,202,248
9,Decreto Supremo N.° 027-2023-PCM,120,120


In [19]:
# Corrección manual: la página del DS 036-2023-PCM no trae el título real,
# solo "Declaratoria de Estado de Emergencia" + el texto del PDF.
# Revisamos el PDF de la norma y copiamos el título de su primera página.
fila = decretos["numero"].str.contains("036-2023-PCM")
decretos.loc[fila, "titulo_completo"] = (
    "Decreto Supremo que declara el Estado de Emergencia en los distritos de Ancón, "
    "Pucusana, Punta Hermosa, Punta Negra, San Bartolo y Santa María del Mar de la "
    "provincia y departamento de Lima, por impacto de daños ante intensas "
    "precipitaciones pluviales"
)
print("Filas corregidas:", fila.sum())

Filas corregidas: 1


## 7. Clasificar: `es_lluvia`, `tipo` y `motivo`

Usamos el título completo en minúsculas y el operador `in`. El **orden** de las condiciones en `tipo` importa: un decreto de prórroga suele decir *"prorrogan el Estado de Emergencia **declarado** mediante..."*, así que primero preguntamos por prórroga y solo después por "declara".

In [20]:
def es_lluvia(titulo):
    t = titulo.lower()
    return ("lluvia" in t) or ("precipitaciones" in t)


def tipo_norma(titulo):
    t = titulo.lower()
    if "prórroga" in t or "prorroga" in t:
        return "prorroga"
    elif "declara" in t:
        return "declara"
    else:
        return "otro"


def motivo_norma(titulo):
    t = titulo.lower()
    if "peligro inminente" in t:
        return "peligro inminente"
    elif "impacto de daños" in t:
        return "impacto de daños"
    else:
        return "otro"


decretos["es_lluvia"] = decretos["titulo_completo"].apply(es_lluvia)
decretos["tipo"] = decretos["titulo_completo"].apply(tipo_norma)
decretos["motivo"] = decretos["titulo_completo"].apply(motivo_norma)

print(decretos["es_lluvia"].value_counts(), "\n")
print(decretos["tipo"].value_counts(), "\n")
print(decretos["motivo"].value_counts())

es_lluvia
False    28
True     21
Name: count, dtype: int64 

tipo
declara     28
prorroga    21
Name: count, dtype: int64 

motivo
otro                 27
impacto de daños     17
peligro inminente     5
Name: count, dtype: int64


### Revisar los que quedaron como `"otro"`

In [21]:
print("=== tipo == 'otro' ===")
for _, f in decretos[decretos["tipo"] == "otro"].iterrows():
    print(f"- {f['numero']}: {f['titulo_completo'][:250]}\n")

print("=== motivo == 'otro' ===")
for _, f in decretos[decretos["motivo"] == "otro"].iterrows():
    print(f"- {f['numero']}: {f['titulo_completo'][:250]}\n")

# ¿Cuáles de los motivo="otro" son de lluvias y pasan al paso 11?
otros = decretos[decretos["motivo"] == "otro"]
print(otros["es_lluvia"].value_counts(), "\n")
print("=== motivo 'otro' que SÍ son de lluvias ===")
for _, f in otros[otros["es_lluvia"]].iterrows():
    print(f"- {f['numero']}: {f['titulo_completo']}\n")

=== tipo == 'otro' ===
=== motivo == 'otro' ===
- Decreto Supremo N.° 016-2023-PCM: Decreto Supremo que declara el Estado de Emergencia en la provincia de Padre Abad del departamento de Ucayali

- Decreto Supremo N.° 010-2023-PCM: Decreto Supremo que declara el Estado de Emergencia en los departamentos de Amazonas La Libertad y Tacna

- Directiva N.° 005-2015-PCM/SGRD: "Directiva para la ejecución del ejercicio de simulación por el Fenómeno El Niño, en los departamentos declarados en Estado de Emergencia". (Resolución Ministerial N° 248-2015-PCM) Esta norma pertenece al compendio Directivas Directiva Nº 005-2015-PC

- Directiva N.° 004-2015-PCM/SGRD: "Directiva para la ejecución de simulacro nocturno por el Fenómeno El Niño, en los departamentos declarados en Estado de Emergencia". (Resolución Ministerial N° 248-2015-PCM) Esta norma pertenece al compendio Directivas Directiva Nº 004-2015-PCM/SGRD

- Decreto Supremo N.° 009-2023-PCM: Decreto Supremo que declara el Estado de Emergencia e

### ¿Están bien clasificados los `"otro"`?

- **`tipo = "otro"`:** no quedó ninguna norma. Todas son `declara` o `prorroga`, y eso es correcto: como `in` busca subcadenas, `"declara"` también encuentra *"Declaran"* y *"Declaratoria"* (DS 036), y `"prorroga"` encuentra *"Prorrogar"* (DS 050). El orden de las condiciones también funcionó: el DS 031 dice *"prorroga el Estado de Emergencia **declarado**..."* y quedó como `prorroga`, no como `declara`.

- **`motivo = "otro"`:** son **28 normas**. Las separamos según `es_lluvia`:
  1. **26 no son de lluvias** (`es_lluvia = False`): emergencias en carreteras de la Red Vial Nacional (DS 032, 051), el Corredor Vial Sur (DS 060), Padre Abad (DS 016, 027), Putumayo (DS 023, 050), entre otras. Sus títulos no mencionan lluvias ni precipitaciones, así que el paso 11 las elimina y su `motivo` no afecta el análisis. Aquí también están dos **Directivas de 2015** (005 y 004-2015-PCM/SGRD), que la búsqueda dejó pasar aunque no son decretos de nuestra temporada.
  2. **2 sí son de lluvias** (`es_lluvia = True`): el DS 043-2023-PCM y su prórroga, el DS 065-2023-PCM. Declaran el *"Estado de Emergencia Nacional, por desastre de gran magnitud"* en Lambayeque, Piura y Tumbes. No dicen "peligro inminente" ni "impacto de daños", así que `"otro"` es la clasificación correcta: es una tercera categoría real, no un error del código.
  3. **Un caso que no se puede clasificar con el título:** el DS 036-2023-PCM. Su página solo trae *"Declaratoria de Estado de Emergencia"* más el texto del PDF, sin el motivo ni los lugares, por eso quedó con `es_lluvia = False`. Revisamos el PDF y [COMPLETAR: sí / no es por lluvias], así que [lo corregimos a mano / lo dejamos fuera].

- **Conclusión:** la clasificación es correcta. En las normas de lluvias, `"otro"` corresponde a los decretos de "desastre de gran magnitud" (DS 043 y 065). Si en la Parte 3 se analiza el motivo, conviene tratarlos como una categoría aparte o renombrarlos como `"desastre de gran magnitud"`. El único caso problemático fue el DS 036, porque su página no trae el título completo.

## 8. Solo normas de lluvias

In [22]:
lluvias = decretos[decretos["es_lluvia"]].reset_index(drop=True)
print("Normas por lluvias:", len(lluvias), "de", len(decretos))
lluvias[["numero", "fecha", "tipo", "motivo", "titulo_completo"]].head(5)

Normas por lluvias: 21 de 49


,numero,fecha,tipo,motivo,titulo_completo
0,Decreto Supremo N.° 008-2023-PCM,13 de enero de 2023,declara,peligro inminente,Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos d...
1,Decreto Supremo N.° 007-2023-PCM,13 de enero de 2023,declara,impacto de daños,Decreto Supremo que declara el Estado de Emergencia en algunos distritos de varias provincias de los departamentos d...
2,Decreto Supremo N.° 024-2023-PCM,17 de febrero de 2023,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Allauca, Cacra, Catahuasi, Huangascar, Linch..."
3,Decreto Supremo N.° 019-2023-PCM,6 de febrero de 2023,declara,impacto de daños,Decreto Supremo que declara el Estado de Emergencia en el distrito de Mariano Nicolás Valcárcel de la provincia de C...
4,Decreto Supremo N.° 044-2023-PCM,29 de marzo de 2023,declara,impacto de daños,Decreto Supremo que declara el Estado de Emergencia en algunos distritos de las provincias de Leoncio Prado y Marañó...


## 9. ¿Qué departamentos menciona cada decreto?

Decisiones para no contar mal:

1. **Buscamos en el título original (con mayúsculas)**, no en minúsculas: así `"Ica"` no aparece dentro de `"huancavelica"`.
2. **Palabra completa** con `\b`: `re.search(r"\b" + departamento + r"\b", titulo)`. `\b` en Python 3 entiende letras con tilde, así que `"Huancavelica"` no activa `\bIca\b`.
3. **Tildes opcionales:** algunos títulos escriben "Ancash", "Junin" o "San Martin" sin tilde. Convertimos cada nombre en un patrón donde la vocal acentuada acepta las dos formas (`Á` → `[ÁA]`).
4. **Cada departamento cuenta una sola vez por decreto:** la pregunta es "¿lo menciona?" (sí/no), no "¿cuántas veces?". Por eso usamos `re.search` (para en la primera coincidencia) y guardamos una lista sin repetidos. Si dice *"provincia de Ica del departamento de Ica"*, Ica queda **una sola vez**.

In [23]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
"Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
"Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
"Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
"San Martín", "Tacna", "Tumbes", "Ucayali"]

TILDES = {"á": "[áa]", "é": "[ée]", "í": "[íi]", "ó": "[óo]", "ú": "[úu]", "Á": "[ÁA]"}

def patron_departamento(nombre):
    p = "".join(TILDES.get(c, re.escape(c)) for c in nombre)
    return re.compile(r"\b" + p + r"\b")

PATRONES = {d: patron_departamento(d) for d in departamentos}

def departamentos_en(titulo):
    return [d for d, patron in PATRONES.items() if patron.search(titulo)]


# Pruebas rápidas de los casos problemáticos
print(departamentos_en("distritos del departamento de Huancavelica"))           # solo Huancavelica
print(departamentos_en("provincia de Ica del departamento de Ica"))             # Ica una sola vez
print(departamentos_en("departamentos de Ancash, Junin y San Martin"))          # sin tildes

['Huancavelica']
['Ica']
['Áncash', 'Junín', 'San Martín']


In [24]:
lluvias["departamentos"] = lluvias["titulo_completo"].apply(departamentos_en)
lluvias["n_departamentos"] = lluvias["departamentos"].str.len()

print("Decretos sin ningún departamento detectado:", (lluvias["n_departamentos"] == 0).sum())
lluvias[["numero", "tipo", "departamentos"]].head(10)

Decretos sin ningún departamento detectado: 0


,numero,tipo,departamentos
0,Decreto Supremo N.° 008-2023-PCM,declara,"[Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Huancavelica, Huánuco, Junín, La Libertad, Lima, Loreto, Moquegua,..."
1,Decreto Supremo N.° 007-2023-PCM,declara,"[Amazonas, Áncash, Cusco, Huánuco, La Libertad, San Martín, Ucayali]"
2,Decreto Supremo N.° 024-2023-PCM,declara,[Lima]
3,Decreto Supremo N.° 019-2023-PCM,declara,[Arequipa]
4,Decreto Supremo N.° 044-2023-PCM,declara,"[Huánuco, Ica]"
5,Decreto Supremo N.° 043-2023-PCM,declara,"[Lambayeque, Piura, Tumbes]"
6,Decreto Supremo N.° 0039-2023-PCM,declara,"[Amazonas, Ayacucho, Pasco, Ucayali]"
7,Decreto Supremo N.° 038-2023-PCM,declara,"[Áncash, Huancavelica, Ica, Lima]"
8,Decreto Supremo N.° 036-2023-PCM,declara,[Lima]
9,Decreto Supremo N.° 035-2023-PCM,declara,"[Áncash, Apurímac, Arequipa, Ayacucho, Callao, Cusco, Huancavelica, Huánuco, Ica, Junín, Lima, Moquegua, Puno, Tacna]"


In [25]:
# Casos para la explicación: decretos que mencionan Ica y/o Huancavelica
mask = lluvias["titulo_completo"].str.contains(r"\bIca\b|Huancavelica", regex=True)
for _, f in lluvias[mask].iterrows():
    print(f["numero"])
    print("  título:", f["titulo_completo"][:300])
    print("  veces que aparece 'Ica' como palabra:", len(re.findall(r"\bIca\b", f["titulo_completo"])))
    print("  departamentos detectados:", f["departamentos"], "\n")

# Si quedaron decretos sin departamento, mirarlos
for _, f in lluvias[lluvias["n_departamentos"] == 0].iterrows():
    print("SIN DEPARTAMENTO →", f["numero"], ":", f["titulo_completo"][:300])

Decreto Supremo N.° 008-2023-PCM
  título: Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Huancavelica, Huánuco, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, San Martín, Tacna y Ucayali, por peligro inminente ante int
  veces que aparece 'Ica' como palabra: 0
  departamentos detectados: ['Amazonas', 'Áncash', 'Arequipa', 'Ayacucho', 'Cajamarca', 'Huancavelica', 'Huánuco', 'Junín', 'La Libertad', 'Lima', 'Loreto', 'Moquegua', 'Pasco', 'San Martín', 'Tacna', 'Ucayali'] 

Decreto Supremo N.° 044-2023-PCM
  título: Decreto Supremo que declara el Estado de Emergencia en algunos distritos de las provincias de Leoncio Prado y Marañón del departamento de Huánuco; y, en algunos distritos de las provincias de Nasca, Ica y Palpa del departamento de Ica, por impacto de daños a consecuencia de intensas precipitaciones 
  veces que aparece 'Ica' como palabra: 2
  departamento

### Ejemplo de nuestra temporada: cómo evitamos contar mal

**Problema 1: "ica" dentro de "Huancavelica".** El *Decreto Supremo N.° 008-2023-PCM* dice *"...en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, **Huancavelica**, Huánuco, Junín, ..."*. Este decreto **no menciona Ica**.
- Si hubiéramos buscado `"ica" in titulo.lower()`, habría contado también para Ica, porque `"huancavelica"` termina en `"ica"`.
- Como buscamos en el título original y por palabra completa (`\bIca\b`), `"Ica"` aparece **0 veces** y en `departamentos` sale solo Huancavelica, que es lo correcto.

**Problema 2: Ica como provincia y como departamento.** El *Decreto Supremo N.° 044-2023-PCM* dice *"...en algunos distritos de las provincias de Nasca, **Ica** y Palpa del departamento de **Ica**..."*.
- `"Ica"` aparece **2 veces** como palabra: una por la provincia y otra por el departamento.
- En `departamentos` sale **una sola vez** (`['Huánuco', 'Ica']`), porque `re.search` solo responde si el departamento aparece o no, y no cuenta cuántas veces. Así, en `decretos_por_departamento.csv` este decreto suma **una** declaratoria a Ica y no dos.
- Que la provincia Ica cuente para el departamento Ica no es un error: la provincia está dentro de ese mismo departamento.

Con el DS 038 (*"departamentos de Áncash, Huancavelica, Ica y Lima"*) comprobamos el caso contrario: cuando el decreto **sí** menciona los dos, se detectan **Huancavelica e Ica** por separado.

**Decretos sin departamento detectado:** no hubo ninguno (`Decretos sin ningún departamento detectado: 0`). Todos los títulos nombran al menos un departamento, aunque declaren la emergencia solo en algunos distritos o provincias. El Callao también se detecta, porque los títulos dicen *"Provincia Constitucional del Callao"*.

## 10. Guardar los archivos

1. `datos/decretos_lluvias.csv`: una fila por norma de lluvias. Guardamos como `titulo` el **título completo** (no el cortado).
2. `datos/decretos_por_departamento.csv`: con `explode` pasamos a una fila por (decreto, departamento) y contamos cuántas declaratorias y prórrogas tiene cada departamento. Incluimos los 25 departamentos, aunque tengan 0, para que el cruce de la Parte 3 no pierda ninguno.

In [26]:
salida = lluvias.copy()
salida["titulo"] = salida["titulo_completo"]
salida = salida[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]]

salida.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")
print("Guardado datos/decretos_lluvias.csv:", salida.shape)
salida.head()

Guardado datos/decretos_lluvias.csv: (21, 6)


,numero,fecha,titulo,tipo,motivo,enlace
0,Decreto Supremo N.° 008-2023-PCM,13 de enero de 2023,Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos d...,declara,peligro inminente,https://www.gob.pe/institucion/pcm/normas-legales/3888261-008-2023-pcm
1,Decreto Supremo N.° 007-2023-PCM,13 de enero de 2023,Decreto Supremo que declara el Estado de Emergencia en algunos distritos de varias provincias de los departamentos d...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/3888259-007-2023-pcm
2,Decreto Supremo N.° 024-2023-PCM,17 de febrero de 2023,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Allauca, Cacra, Catahuasi, Huangascar, Linch...",declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/5569220-024-2023-pcm
3,Decreto Supremo N.° 019-2023-PCM,6 de febrero de 2023,Decreto Supremo que declara el Estado de Emergencia en el distrito de Mariano Nicolás Valcárcel de la provincia de C...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/3900649-019-2023-pcm
4,Decreto Supremo N.° 044-2023-PCM,29 de marzo de 2023,Decreto Supremo que declara el Estado de Emergencia en algunos distritos de las provincias de Leoncio Prado y Marañó...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/4116711-044-2023-pcm


In [27]:
largo = lluvias[["numero", "tipo", "departamentos"]].explode("departamentos")
largo = (largo.dropna(subset=["departamentos"])
              .rename(columns={"departamentos": "departamento"})
              .reset_index(drop=True))          # explode repite el índice

por_depto = (
    pd.crosstab(largo["departamento"], largo["tipo"])
      .reindex(departamentos, fill_value=0)          # los 25, con 0 si no aparecen
)
for col in ["declara", "prorroga"]:
    if col not in por_depto.columns:
        por_depto[col] = 0

por_depto = (por_depto
             .rename(columns={"declara": "declaratorias", "prorroga": "prorrogas"})
             [["declaratorias", "prorrogas"]]
             .reset_index()
             .rename(columns={"index": "departamento"}))
por_depto.columns.name = None

por_depto.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
print("Guardado datos/decretos_por_departamento.csv:", por_depto.shape)
por_depto.sort_values("declaratorias", ascending=False)

Guardado datos/decretos_por_departamento.csv: (25, 3)


,departamento,declaratorias,prorrogas
14,Lima,7,0
1,Áncash,6,1
12,La Libertad,5,0
4,Ayacucho,5,1
21,San Martín,4,0
3,Arequipa,4,0
5,Cajamarca,4,0
9,Huánuco,4,0
19,Piura,3,1
13,Lambayeque,3,1


In [28]:
# Verificación final: la suma por departamento debe cuadrar con el explode
print("Filas (decreto, depto) declara :", (largo["tipo"] == "declara").sum(),
      "| suma declaratorias:", por_depto["declaratorias"].sum())
print("Filas (decreto, depto) prórroga:", (largo["tipo"] == "prorroga").sum(),
      "| suma prórrogas     :", por_depto["prorrogas"].sum())

# Releer los CSV para comprobar que se guardaron bien (tildes incluidas)
pd.read_csv("datos/decretos_por_departamento.csv").head()

Filas (decreto, depto) declara : 73 | suma declaratorias: 73
Filas (decreto, depto) prórroga: 6 | suma prórrogas     : 6


,departamento,declaratorias,prorrogas
0,Amazonas,3,1
1,Áncash,6,1
2,Apurímac,1,0
3,Arequipa,4,0
4,Ayacucho,5,1


---
## Resumen

| Paso | Qué hicimos | Verificación |
|---|---|---|
| robots.txt | lo imprimimos y lo probamos con `robotparser` | `/busquedas` permitido, pausas por `Crawl-delay` |
| Selenium | un mes, luego enero–mayo 2023, `WebDriverWait` + pausa ≥ 2 s | tabla `resultados_totales` = `resultados_extraidos` |
| Limpieza | duplicados y normas que no son de `/institucion/pcm/` | se imprime cuántas se quitaron |
| Título completo | `requests` + `BeautifulSoup`, `div.description`, pausa 1 s | ningún título termina en "..." |
| Clasificación | `es_lluvia`, `tipo`, `motivo` con `in` | revisión manual de los `"otro"` |
| Departamentos | regex `\b...\b` sobre el título original, una vez por decreto | casos Ica / Huancavelica |